# NGS: SNP calling, genotype calling and imputation

In this exercise we will work with **30 European (CEU) samples** that have been sequenced to **low
depth** - a few reads per site - on a 3 Mb piece of chromosome 20.

At this depth many sites in an individual are covered by one read, two reads, or no read at all. A
single read can never tell you whether the individual is homozygous or heterozygous, and even three
reads from a true heterozygote all show the same allele about 25% of the time. Calling genotypes one
individual and one site at a time is therefore a losing game. The way out is to *share information*:

 - between **sites**, because nearby alleles are inherited together on haplotypes (linkage disequilibrium),
 - between **individuals**, because unrelated individuals share short haplotype stretches,
 - and with an external **reference panel** of high quality haplotypes.

This is exactly what imputation does, and this exercise is about how much it helps.

### What you will do

 1. Look at the raw low depth data in pileup format.
 2. Call SNPs and genotypes **without** imputation (bcftools and ANGSD).
 3. Impute genotypes with three different strategies (Beagle 4.1, QUILT2, Beagle 5.5).
 4. Compare all of the results to a **truth data set** - the same individuals sequenced to high depth.
 5. Bonus: impute a mother (and think about her fetus) from simulated NIPT data.

Along the way there are short **follow-up questions** after most code cells. They are the point of the
exercise - the commands themselves are just a few lines of copy-paste.

### The software we will use

The first two tools only *call* - they use the reads of one individual (plus, for ANGSD, the estimated
allele frequency) and nothing else. The last three tools *impute* - they model the data as a mosaic of
haplotypes and therefore also use the sites around the site of interest.

| Tool | Input | Works on low depth data? | Needs a reference panel? | Output |
| --- | --- | --- | --- | --- |
| **bcftools** call | BAM (pileup) | No - a low depth genotype call is little more than a guess | No | called genotypes (GT), GL/GQ/GP, only at the sites it calls variable |
| **ANGSD** | BAM | For genotype *likelihoods* yes, for genotype *calls* no | No (uses the estimated allele frequency as prior) | genotype likelihoods, allele frequencies, called genotypes |
| **Beagle 4.1** (`gl=`) | genotype likelihoods (VCF/BCF) | Yes | Optional - here we use **no** panel, the 33 study samples impute each other | phased genotypes, GP, dosage |
| **QUILT2** | BAM | Yes - it is designed for 0.1-1x data | Yes (required) | phased genotypes, GP, DS, HD |
| **Beagle 5.5** (`gt=`) | **called** genotypes (e.g. a SNP chip) | No - it needs confident genotypes as input | Yes (required) | phased genotypes at all panel sites, GP |

Two things are worth noticing already now:

 - Beagle 4.1 and QUILT2 read *uncertain* data (likelihoods or reads), while Beagle 5.5 reads *hard*
   genotype calls. Feeding low depth genotype calls to Beagle 5.5 would mean feeding it a lot of
   confidently wrong genotypes.
 - Only Beagle 5.5 and QUILT2 know in advance which sites are variable, because they are given a
   reference panel. bcftools, ANGSD and Beagle 4.1 have to discover the SNPs themselves.

For the SNP/genotype callers the default behaviour also differs quite a lot:

| | SAMtools/bcftools | GATK | ANGSD |
| --- | --- | --- | --- |
| Genotype likelihood | tries to model error dependencies | simple | user specified |
| SNP caller | uses the allele frequency (SFS) as prior | heterozygosity prior 0.1% | likelihood ratio test |
| Genotype caller | allele frequency prior | maximum likelihood | allele frequency prior |

All of the programs have many additional filters and options - we only look at close-to-default behaviour.

## 1. Setup

The first cell only defines paths. To make sure that nothing breaks if you restart the kernel or run
the cells out of order, it writes all the paths to a small file (`env.sh`) and every later bash cell
starts by reading that file back in with `source`. The file also does `cd` into your working folder,
so all later commands can use short relative paths like `bams/` and `vcfs/`.

In [ ]:
# ---------------------------------------------------------------------------
# where the course data and the software is (only change this if the course
# files have been moved to another folder)
# ---------------------------------------------------------------------------
COURSE_PATH=/course/popgen25
DATA_PATH=$COURSE_PATH/Imputation
SOFTWARE_PATH=$COURSE_PATH/software

# our own working folder
WORK=~/advBinfImputation
mkdir -p $WORK

# ---------------------------------------------------------------------------
# write the paths to a file so that every later cell can get them back with
# a single "source ~/advBinfImputation/env.sh"
# ---------------------------------------------------------------------------
cat > $WORK/env.sh <<EOF
# ---- data and software ----
export DATA_PATH=$DATA_PATH
export SOFTWARE_PATH=$SOFTWARE_PATH
export WORK=$WORK

# ---- java programs and the QUILT2 R script ----
export BEAGLE4=$SOFTWARE_PATH/beagle.27Jan18.7e1.jar
export BEAGLE5=$SOFTWARE_PATH/beagle.27Feb25.75f.jar
export QUILT2=$SOFTWARE_PATH/QUILT/QUILT2.R

# ---- reference files (relative to the working folder) ----
export REF_GENOME=resources/GRCh38_full_analysis_set_plus_decoy_hla.fa
export QUILT2_MAP=resources/CEU-chr20-final.b38.txt.gz
export BEAGLE5_MAP=resources/plink.chr20.GRCh38.rename.map

# ---- vcf files ----
export REF_VCF=vcfs/CEU_ref_set.chr20.vcf.gz
export TRUE_VCF=vcfs/CEU_true_set.chr20.vcf.gz
export FAKE_SNPCHIP=vcfs/CEU_fake_chip.chr20.vcf.gz
export QUILT2_EXAMPLE=vcfs/quilt2_all_inds.vcf.gz

# ---- the region we analyse in this exercise ----
export REGION=chr20:2000001-5000000

# always work from the same folder
cd $WORK
EOF

echo "--- wrote $WORK/env.sh ---"
cat $WORK/env.sh

Now link the data into the working folder and check that all programs and files are where we expect
them to be. **Make sure there are no error messages in the output of the next cell** - if a file is
missing, everything below will fail.

In [ ]:
source ~/advBinfImputation/env.sh   # paths + cd to the working folder

# make the data visible from our working folder (symbolic links, no copying)
ln -sfn $DATA_PATH/resources .
ln -sfn $DATA_PATH/vcfs .
ln -sfn $DATA_PATH/bams .

# the small json files with the quiz questions
cp -sf $DATA_PATH/quiz/*.json .

echo "--- programs that are installed ---"
type angsd samtools bcftools vcftools java

echo -e "\n--- java tools and the QUILT2 script ---"
ls -l $BEAGLE4 $BEAGLE5 $QUILT2

echo -e "\n--- reference files, maps and vcfs ---"
ls -lL $REF_GENOME $QUILT2_MAP $BEAGLE5_MAP $REF_VCF $TRUE_VCF $FAKE_SNPCHIP $QUILT2_EXAMPLE

echo -e "\n--- our working folder ---"
pwd
ls

There are a few quizzes along the way. They are run from Python, so we also need to point the
Python kernel at the working folder.

In [ ]:
# set up the python working space
import os

work_d = os.path.expanduser("~/advBinfImputation")
os.chdir(work_d)
print(work_d)

**Follow-up**

 - Which of the files above is the *reference panel* and which one is the *truth*? (hint: look at
   `CEU_ref_set` and `CEU_true_set`)
 - Why must the truth set and the reference panel contain **different** individuals?

## 2. The data

We start from BAM files - aligned reads - one per individual. Most programs take a *bamlist*: a plain
text file with one path per line.

In [ ]:
source ~/advBinfImputation/env.sh

# one line per bam file
ls bams/NA*.bam > CEU_inds_bams.list

echo "--- number of individuals ---"
wc -l < CEU_inds_bams.list

echo -e "\n--- first lines of the bamlist ---"
head -n 3 CEU_inds_bams.list

**Follow-up**

 - How many individuals are in the bamlist? (use `wc` and `cat` on the file - please do **not** use `less`)
 - There is one more bam file in the `bams/` folder that we did *not* put in the list. Which one, and
   why do you think it is kept separate? (`ls bams/`)

### How much data do we actually have?

`samtools coverage` gives a quick summary of a bam file in a region.

In [ ]:
source ~/advBinfImputation/env.sh

echo "--- coverage in $REGION for the first three individuals ---"
for BAM in $(head -n 3 CEU_inds_bams.list)
do
    samtools coverage -r $REGION $BAM | tail -n +2 | awk -v b=$(basename $BAM) '{print b, "covered_bases_fraction:", $6"%", "mean_depth:", $7}'
done

**Follow-up**

 - What is the mean depth per individual? What fraction of the region is covered by at least one read?
 - If a site is covered by exactly one read carrying the reference allele, what are the possible true
   genotypes? Can you tell them apart from that single read?
 - Add up the reads of all 30 individuals at one site. That sum is the information that allele
   frequency estimation (and imputation) can use even when not a single genotype is known. Compare it
   to what you know about *one* individual at that site.

### The pileup format

The pileup format lines up the reads of all individuals site by site. This is what all the callers
see internally.

In [ ]:
source ~/advBinfImputation/env.sh

# only the first 9 columns are printed to keep the output readable
samtools mpileup -b CEU_inds_bams.list --region chr20:2000600-2000620 2>/dev/null | cut -f 1-9

**Follow-up**

 - Identify the columns: what are the first three columns, and what do the following triplets of
   columns contain? (change the `cut` above, or the region, if you want to see more)
 - How many columns does the full output have for 30 individuals?
 - Find a site where an individual has more than one read. Do the reads agree?

## 3. SNP and genotype calling **without** imputation

Calling has two steps that are easy to mix up:

 1. **SNP calling / variant discovery**: is this site variable in my samples at all?
 2. **Genotype calling**: given that the site is variable, what is the genotype of each individual?

Both steps are done from genotype likelihoods `P(reads | genotype)`, but they use different priors,
and at low depth the prior does most of the work. Let us see what two very different programs do.

### 3a. bcftools

`bcftools mpileup` computes genotype likelihoods from the bams, and `bcftools call` turns them into
SNP calls and genotype calls. First have a look at the options.

In [ ]:
source ~/advBinfImputation/env.sh

# the usage is printed to stderr, so redirect it and show the first lines
bcftools mpileup 2>&1 | head -n 25

Now call SNPs and genotypes in our 3 Mb region. This takes about 3 minutes, so read on while it
runs: `-V indels` skips indels, `-m` is the multiallelic caller, `-v` only outputs variable sites and
`-a GQ,GP` asks for genotype qualities and genotype posterior probabilities.

In [ ]:
source ~/advBinfImputation/env.sh
# ~3 min to run
mkdir -p bcftoolsgt_out

bcftools mpileup -f $REF_GENOME -Ou -b CEU_inds_bams.list -r $REGION | \
    bcftools call -V indels -a GQ,GP -mv -Ov -o bcftoolsgt_out/CEU_inds_bam.vcf

# compress and index the vcf file
bgzip -f bcftoolsgt_out/CEU_inds_bam.vcf
tabix -f bcftoolsgt_out/CEU_inds_bam.vcf.gz

echo "--- created files ---"
ls -l bcftoolsgt_out

**Follow-up**

 - What does the `-f` option do, and why does a genotype caller need the reference genome?
 - What is the difference between `GQ`, `GP` and `GL`/`PL`?
 - Why did we ask for `-v` (variable sites only)? What would the file look like without it?

In [ ]:
source ~/advBinfImputation/env.sh

echo "--- number of variable sites called by bcftools ---"
bcftools view -H bcftoolsgt_out/CEU_inds_bam.vcf.gz | wc -l

echo -e "\n--- the last header line and the first 3 variants ---"
echo "(columns: CHROM POS ID REF ALT FORMAT and the first three individuals)"
bcftools view -h bcftoolsgt_out/CEU_inds_bam.vcf.gz | tail -n 1 | cut -f 1-5,9-12

# awk instead of head, so that bcftools is not killed by a broken pipe
bcftools view -H bcftoolsgt_out/CEU_inds_bam.vcf.gz | awk 'NR<=3' | cut -f 1-5,9-12

**Follow-up**

 - Which field of the per-sample columns is the genotype? What does `./.` mean, and why are there so
   many of them at 1x depth?
 - Look at the `GP`/`GQ` values of a called heterozygote. Would you trust it?

### 3b. ANGSD

ANGSD keeps the uncertainty: its main output is the **genotype likelihood** for every site and every
individual. It can also call genotypes, and here we ask it to do both. Note the priors:

 - SNPs are called with a **likelihood ratio test** (`-SNP_pval 0.001`) on the estimated allele frequency,
 - the allele frequency is estimated with an EM algorithm (`-doMaf 2`) from the genotype likelihoods,
 - genotypes are called from the **posterior** (`-doPost 2`), i.e. using the estimated allele
   frequency (and Hardy-Weinberg) as prior.

By default ANGSD picks the major and minor allele from the data. We use `-doMajorMinor 4` so that the
reference allele is forced to be the major allele - this makes the output directly comparable to
bcftools and to the truth set.

In [ ]:
source ~/advBinfImputation/env.sh
# ~3 min to run
mkdir -p angsd_out

angsd -bam CEU_inds_bams.list -SNP_pval 0.001 \
 -doMaf 2 -doMajorMinor 4 -r $REGION \
 -doGlf 2 -doGeno 2 -doPost 2 -GL 1 -doBcf 1 -P 4 \
 -out angsd_out/angsd_genotype -ref $REF_GENOME

bcftools index -f angsd_out/angsd_genotype.bcf

echo "--- created files ---"
ls -l angsd_out

**Follow-up**

 - `-GL 1` is the SAMtools genotype likelihood model. Which part of the command decides which sites
   end up in the output, and which part decides the genotypes?
 - You can see all options with `angsd -bam`, `angsd -doMaf`, `angsd -doGeno` (or at
   <https://www.popgen.dk/angsd/index.php/ANGSD>). What would `-doPost 1` have done instead of `-doPost 2`?

Select the text below to see how to read the command:

<span style="color: white">We analyse the bam files in 'CEU_inds_bams.list' limited to a region of chr20. We only keep sites
that look variable in a likelihood ratio test with p-value 0.001. We estimate the allele frequency
(which requires a major and a minor allele), and we call genotypes from the posterior probability
which uses that frequency as a prior. All of it is based on the SAMtools model for genotype likelihoods.</span>

In [ ]:
source ~/advBinfImputation/env.sh

echo "--- estimated allele frequencies (first lines) ---"
zcat angsd_out/angsd_genotype.mafs.gz | awk 'NR<=5'

echo -e "\n--- genotype likelihood file, first 2 sites, first 4 individuals ---"
zcat angsd_out/angsd_genotype.beagle.gz | awk 'NR<=3' | cut -f 1-15

echo -e "\n--- number of sites angsd called variable ---"
zcat angsd_out/angsd_genotype.mafs.gz | tail -n +2 | wc -l

**Follow-up**

 - The genotype likelihood file has three numbers per individual per site. What are they, and why are
   they normalised so that they sum to one?
 - Compare the number of sites found by ANGSD and by bcftools. Which program is more liberal?
 - The `mafs.gz` file has a column with the estimated frequency. How can a frequency be estimated
   with 1x data when almost no genotype is known?

### Quiz

In [ ]:
from jupyterquiz import display_quiz
display_quiz('call_genotypes_quiz1.json')

### 3c. Comparing SNP discovery

Before we look at genotypes, let us ask the simpler question: **do the two callers find the same
variable sites, and are those sites really variable?** We have a truth set (the same individuals
sequenced to high depth), so we can check.

This uses R with the `vcfppR` package (fast VCF reading) and `UpSetR` (for plotting set overlaps).

In [ ]:
library(vcfppR)
library(UpSetR)

# helper: pull out a matrix (e.g. genotypes) from one vcf, with the rows (sites)
# and columns (individuals) matched to the truth set
find_match <- function(vcf_in, vcf_truth, name="tool", data_type="gt"){
    if(!all(vcf_truth$samples %in% vcf_in$samples)){stop("samples doesn't match")}
    true_pos_match <- match(vcf_truth$pos, vcf_in$pos)
    vcf_in[[data_type]][true_pos_match, match(vcf_truth$samples, vcf_in$samples)]
}

# work in the same folder as the bash cells
(work_d <- path.expand("~/advBinfImputation"))
setwd(work_d)

region <- "chr20:2000001-5000000"

res_truth    <- vcftable("vcfs/CEU_true_set.chr20.vcf.gz", region, vartype = "snps")
res_bcftools <- vcftable("bcftoolsgt_out/CEU_inds_bam.vcf.gz", region, vartype = "snps")
res_angsd    <- vcftable("angsd_out/angsd_genotype.bcf", region, vartype = "snps")

# the bam based files use file names as sample names - strip them down to the sample id
res_truth$samples    <- sub(".lc.bam", "", basename(res_truth$samples))
res_bcftools$samples <- sub(".lc.bam", "", basename(res_bcftools$samples))
res_angsd$samples    <- sub(".lc.bam", "", basename(res_angsd$samples))

cat("number of SNPs\n")
c(truth=length(res_truth$pos), bcftools=length(res_bcftools$pos), angsd=length(res_angsd$pos))

**Follow-up**

 - Both callers see exactly the same reads. Why do they not find the same number of SNPs?

An **UpSet plot** is a readable alternative to a Venn diagram: each column is a combination of sets
(the black dots below the bars) and the bar is how many positions fall in exactly that combination.

In [ ]:
upsetdf <- fromList(list(
    Truth    = res_truth$pos,
    angsd    = res_angsd$pos,
    bcftools = res_bcftools$pos
))

upset(
  upsetdf,
  order.by        = "freq",
  sets.x.label    = "SNP count",
  mainbar.y.label = "shared SNP",
  text.scale = c(2.2, 1.8, 2.2, 1.8, 2, 1.8)
)

**Follow-up**

 - Which method is best at *finding* the SNPs?
 - How many **false positives** (called variable but not variable in the truth) does each method have?
 - How many **false negatives** (variable in the truth but not called)? Which kind of SNP do you think
   is missed - common or rare ones?
 - Would you rather have a caller with few false positives or few false negatives, if the next step is
   imputation?

## 4. Genotype calling **with** imputation

An imputation program does not treat the sites independently. It models each individual's two
chromosomes as a **mosaic of haplotypes** (a hidden Markov model, where switching between haplotypes
is a recombination event and the genetic map tells it how likely a switch is). Once you know which
haplotype an individual is copying at a position, you also know which allele to expect - even at a
site with no reads at all.

The haplotypes to copy from can come from

 - the **study samples themselves** (Beagle 4.1 below: 33 samples impute each other), or
 - an external **reference panel** of phased high quality haplotypes (QUILT2 and Beagle 5.5 below).

We will run three variations on this theme and then compare them all against the truth.

### 4a. Beagle 4.1 from genotype likelihoods (no reference panel)

Beagle 4.1 can take **genotype likelihoods** as input (`gl=`), which is exactly what ANGSD produced.
It then estimates the haplotypes and the genotypes jointly from the 33 samples - no reference panel
involved. First convert the ANGSD BCF to an indexed VCF.

In [ ]:
source ~/advBinfImputation/env.sh

mkdir -p beagle4_out
bcftools view angsd_out/angsd_genotype.bcf -Ov -o beagle4_out/angsd_for_beagle.vcf
bgzip -f beagle4_out/angsd_for_beagle.vcf
tabix -f beagle4_out/angsd_for_beagle.vcf.gz

echo "--- created files ---"
ls -l beagle4_out

Now run the imputation. `gl=` tells Beagle that the input is genotype likelihoods (as opposed to
`gt=`, hard genotype calls), and `niterations` is the number of iterations of the phasing/imputation
algorithm.

In [ ]:
source ~/advBinfImputation/env.sh
# ~10 sec to run
java -Xmx8g -jar $BEAGLE4 \
    gl=beagle4_out/angsd_for_beagle.vcf.gz \
    out=beagle4_out/beagle4_imputation \
    niterations=10 \
    nthreads=8

bcftools index -f beagle4_out/beagle4_imputation.vcf.gz

echo "--- created files ---"
ls -l beagle4_out

**Follow-up**

 - What is the reference panel in this run? Where do the haplotypes that Beagle copies from come from?
 - Beagle 4.1 can only impute at sites that are in the input file. What happens to a SNP that ANGSD
   never discovered?
 - Do you expect this to work better for common or for rare variants, when you only have 33 samples?

In [ ]:
source ~/advBinfImputation/env.sh

echo "--- number of sites before and after imputation ---"
echo -n "angsd input:   "; bcftools view -H beagle4_out/angsd_for_beagle.vcf.gz | wc -l
echo -n "beagle output: "; bcftools view -H beagle4_out/beagle4_imputation.vcf.gz | wc -l

echo -e "\n--- first 2 variants (CHROM POS ID REF ALT FORMAT + 3 individuals) ---"
bcftools view -H beagle4_out/beagle4_imputation.vcf.gz | awk 'NR<=2' | cut -f 1-5,9-12

**Follow-up**

 - Are there any missing genotypes (`./.`) left in the Beagle output? Compare with the bcftools VCF -
   what did imputation do to the missingness?
 - The genotypes are now written as `0|1` rather than `0/1`. What does the `|` mean?

### 4b. QUILT2 - straight from BAM files with a reference panel

QUILT2 skips the genotype calling step entirely: it reads the **reads** and a **reference panel** and
returns imputed, phased genotypes. This is currently the best approach for very low depth sequencing
data.

Running all 30 individuals takes about 20 minutes, so we run **one** individual as a demonstration
(2 minutes, most of it spent preparing the reference panel) and then use a precomputed result for all
30 individuals in the comparison below.

In [ ]:
source ~/advBinfImputation/env.sh
# ~2 min to run
mkdir -p quilt2_1_ind

# a bamlist with only the first individual
head -n 1 CEU_inds_bams.list > CEU_1_ind.list
cat CEU_1_ind.list

$QUILT2 \
    --outputdir=quilt2_1_ind \
    --chr=chr20 \
    --regionStart=2000001 \
    --regionEnd=5000000 \
    --buffer=500000 \
    --nGen=100 \
    --bamlist=CEU_1_ind.list \
    --genetic_map_file=$QUILT2_MAP \
    --reference_vcf_file=$REF_VCF \
    --save_prepared_reference=TRUE

echo "--- created files ---"
ls -l quilt2_1_ind

**Follow-up**

 - Which of the inputs above is the reference panel, and which is the genetic map? What is the map
   used for?
 - `--buffer=500000` adds 500 kb on each side of the region. Why is a buffer needed when you impute a
   piece of a chromosome?
 - `--nGen=100` is the number of generations since the panel and the samples shared ancestors. Would
   you use the same value for a European sample imputed from a European panel and from an African panel?
 - QUILT2 saved a prepared reference (`--save_prepared_reference=TRUE`). Why does that make the next
   run much faster?

In [ ]:
source ~/advBinfImputation/env.sh

# The full 30 individual run takes ~20 min. It would be exactly the command above with
# --bamlist=CEU_inds_bams.list, i.e.:
#
# $QUILT2 --outputdir=quilt2_out --chr=chr20 --regionStart=2000001 --regionEnd=5000000 \
#     --buffer=500000 --nGen=100 --bamlist=CEU_inds_bams.list \
#     --genetic_map_file=$QUILT2_MAP --reference_vcf_file=$REF_VCF
#
# instead we copy the precomputed result for all 30 individuals
cp -f $QUILT2_EXAMPLE quilt2_all_inds.vcf.gz
tabix -f quilt2_all_inds.vcf.gz

echo -n "individuals in the precomputed QUILT2 file: "
bcftools query -l quilt2_all_inds.vcf.gz | wc -l
echo -n "sites in the precomputed QUILT2 file:       "
bcftools view -H quilt2_all_inds.vcf.gz 2>/dev/null | wc -l

### The output VCF fields

Imputation output keeps the uncertainty, which is the whole point. Per sample you get:

 - **GT** - phased genotypes, where each allele is the rounded per-haplotype posterior probability (HD below)
 - **GP** - genotype posteriors: the posterior probabilities of the three genotypes given the data
 - **DS** - diploid dosage: the posterior *expected* number of alternative alleles (a number between 0 and 2)
 - **HD** - haploid dosages: the per-haplotype posterior probability of the alternative allele

Note that in QUILT the genotype posteriors (GP) and dosages (DS) come from the main Gibbs sampling,
while the phasing (GT and HD) comes from an extra phasing Gibbs sample. They can therefore be
slightly inconsistent with each other; if you need consistency you can build GP and DS from HD.

In [ ]:
source ~/advBinfImputation/env.sh

# a few lines from the middle of the file (awk, so zcat is not killed by a broken pipe)
zcat quilt2_all_inds.vcf.gz | awk 'NR>=95 && NR<=99' | cut -f 1-12

**Follow-up**

 - Find a site where GT is `0|0` but DS is clearly above 0. What does that tell you about how certain
   the call is?
 - For downstream analyses (e.g. a GWAS or a PCA) would you rather use GT or DS? Why?
 - A site where all individuals have GP close to (1, 0, 0) - is that a well imputed site or an
   uninformative one?

### 4c. Beagle 5.5 - imputing SNP chip data from a large reference panel

The third strategy is the classical one, and the most common one in human genetics: you have a
**SNP chip** with a modest number of confidently called genotypes and you want genotypes at all the
sites in a big reference panel.

To try it out we made a fake SNP chip: we took the truth genotypes and kept only a subset of the
sites, pretending that those are the sites on the chip. Beagle 5.5 then fills in everything else.
Note that this is a *different* (and much easier) problem than the two runs above: the input
genotypes here are essentially error free.

In [ ]:
source ~/advBinfImputation/env.sh

echo "--- sites on the fake SNP chip vs sites in the reference panel ---"
echo -n "chip:  "; bcftools view -H $FAKE_SNPCHIP | wc -l
echo -n "panel: "; bcftools view -H $REF_VCF | wc -l

**Follow-up**

 - What is the ratio between the two numbers? For every genotyped site, how many sites does Beagle
   have to guess?

In [ ]:
source ~/advBinfImputation/env.sh
# ~10 sec to run
mkdir -p beagle5_out

java -Xmx8g -jar $BEAGLE5 \
    gt=$FAKE_SNPCHIP \
    ref=$REF_VCF \
    map=$BEAGLE5_MAP \
    nthreads=8 \
    impute=true \
    gp=true \
    out=beagle5_out/beagle5_imputed

bcftools index -f beagle5_out/beagle5_imputed.vcf.gz

echo -n "--- sites in the beagle 5.5 output: "
bcftools view -H beagle5_out/beagle5_imputed.vcf.gz | wc -l

echo "--- created files ---"
ls -l beagle5_out

**Follow-up**

 - `gt=` instead of `gl=`: what would go wrong if we gave Beagle 5.5 the low depth genotype calls
   from bcftools instead of the SNP chip?
 - The output has many more sites than the input. Which sites are they?
 - What are the three genotype imputation methods above using as input, and what do they return?
   Fill in the table from the top of the notebook from memory.

### Quiz

The numbers are in the output of the cells above - scroll back if you need to.

In [ ]:
from jupyterquiz import display_quiz
display_quiz('call_genotypes_quiz2.json')

## 5. Comparing all approaches against the truth

Now we have five sets of genotypes for (mostly) the same individuals and sites:

| name | how it was made |
| --- | --- |
| `bcftools` | genotype calling, no imputation |
| `angsd` | genotype calling from the posterior, no imputation |
| `beagle4gl` | imputation from genotype likelihoods, no reference panel |
| `quilt2` | imputation from bams with a reference panel |
| `beagle5gl` | imputation of SNP chip genotypes with a reference panel |

The truth set is the same individuals sequenced to high depth. We read all files, match sites and
individuals to the truth, and compute

 - the **missing genotype rate**, and
 - the **concordance rate**: the fraction of genotypes that are identical to the truth. Missing
   genotypes make this ambiguous, so we report it both ways - counting a missing genotype as wrong,
   and ignoring missing genotypes altogether.

In [ ]:
library(vcfppR)

# concordance and missingness for a list of genotype matrices
estimate_gt_tools <- function(gt_mat_list = list(), gt_true_mat = matrix(), keep = c()){
    if(!all(lapply(gt_mat_list, nrow) == nrow(gt_true_mat))){stop("SNP count doesn't match")}
    if(!all(lapply(gt_mat_list, ncol) == ncol(gt_true_mat))){stop("Sample count doesn't match")}
    if(length(keep) == 0){
        keep <- rep(TRUE, nrow(gt_true_mat))
    }

    # missing genotypes counted as discordant
    res <- sapply(gt_mat_list, function(x){
        x[is.na(x)] <- -1
        mean(gt_true_mat[keep, ] == x[keep, ], na.rm=TRUE)})

    # fraction of missing genotypes
    resMis <- sapply(gt_mat_list, function(x)
        mean(is.na(x[keep, ]), na.rm=TRUE))

    # missing genotypes ignored
    resNA <- sapply(gt_mat_list, function(x)
        mean(gt_true_mat[keep, ] == x[keep, ], na.rm=TRUE))

    cat("Missing genotype rate:\n")
    print(round(resMis, 4))
    cat("\nConcordance rate assuming missing is discordant:\n")
    print(round(res, 4))
    cat("\nConcordance rate when ignoring missing genotypes:\n")
    print(round(resNA, 4))
    invisible(list(missing=resMis, concordance=res, concordance_nonmissing=resNA))
}

Read in all the results and match them to the truth set, site by site and individual by individual.

In [ ]:
setwd(path.expand("~/advBinfImputation"))
region <- "chr20:2000001-5000000"

res_truth     <- vcftable("vcfs/CEU_true_set.chr20.vcf.gz", region, vartype = "snps")
res_bcftools  <- vcftable("bcftoolsgt_out/CEU_inds_bam.vcf.gz", region, vartype = "snps")
res_angsd     <- vcftable("angsd_out/angsd_genotype.bcf", region, vartype = "snps")
res_beagle4gl <- vcftable("beagle4_out/beagle4_imputation.vcf.gz", region, vartype = "snps")
res_quilt2    <- vcftable("quilt2_all_inds.vcf.gz", region, vartype = "snps")
res_beagle5gl <- vcftable("beagle5_out/beagle5_imputed.vcf.gz", region, vartype = "snps")

# sample names: strip path and .lc.bam so that all files use the same sample ids
for(nm in c("res_truth","res_bcftools","res_angsd","res_beagle4gl","res_quilt2","res_beagle5gl")){
    x <- get(nm)
    x$samples <- sub(".lc.bam", "", basename(x$samples))
    assign(nm, x)
}

# the truth defines the sites and the order of the individuals
res_truth_mat     <- res_truth$gt
res_bcftools_mat  <- find_match(res_bcftools,  res_truth, name="bcftools")
res_angsd_mat     <- find_match(res_angsd,     res_truth, name="angsd")
res_beagle4gl_mat <- find_match(res_beagle4gl, res_truth, name="beagle4gl")
res_beagle5gl_mat <- find_match(res_beagle5gl, res_truth, name="beagle5gl")
res_quilt2_mat    <- find_match(res_quilt2,    res_truth, name="quilt2")

cat("truth genotype matrix: ", nrow(res_truth_mat), "sites x", ncol(res_truth_mat), "individuals\n")

**Follow-up**

 - `find_match` puts `NA` in the rows of sites that a tool never reported. Is a site that a tool never
   even looked at the same kind of error as a genotype it got wrong? Keep that in mind when you read
   the numbers below.

In [ ]:
gt_list <- list(
    bcftools  = res_bcftools_mat,
    angsd     = res_angsd_mat,
    beagle4gl = res_beagle4gl_mat,
    beagle5gl = res_beagle5gl_mat,
    quilt2    = res_quilt2_mat)

estimate_gt_tools(gt_list, res_truth_mat)

**Follow-up**

 - Which method is best? Does your answer change depending on whether missing genotypes count as
   errors?
 - Why is there such a big difference in missingness between the methods? (which methods were told
   in advance which sites are variable?)
 - The two non-imputation methods look terrible when missing counts as an error and quite good when
   it does not. Both numbers are "true" - which one would you report in a paper, and why?
 - Most sites in the truth set are rare variants where almost everybody is homozygous reference. What
   concordance would you get by not looking at the data at all and always guessing "homozygous
   reference"? Keep that number in mind as the baseline.

### Common vs rare variants

Concordance over *all* sites mixes two very different problems. Most sites in the truth set are rare,
and a rare variant is easy to get right by guessing "homozygous reference". Let us split the sites.

In [ ]:
# allele frequency from the true genotypes
true_freq <- rowMeans(res_truth$gt, na.rm=TRUE)/2

# is the SNP common?
table(true_common <- true_freq > 0.05 & true_freq < 0.95)

**Follow-up**

 - Are most SNPs in the region common or rare?
 - Before you run the next cell: for which methods do you expect the concordance to *increase* when we
   only look at common SNPs, and for which do you expect it to *drop*?

In [ ]:
estimate_gt_tools(gt_list, res_truth_mat, keep=true_common)

**Follow-up**

 - Why did the performance increase for some approaches and drop for others?
 - Which approach "knows" which SNPs are there, and which one has to infer it?
 - Which approach has the best *input* data, and which has the best *method*? Are they the same?
 - If you had money for either (a) sequencing 30 samples at 1x plus a reference panel, or (b)
   sequencing 3 samples at 10x, which would you choose for estimating allele frequencies?

### Performance across allele frequency bins

The comparison above used one arbitrary threshold (5%). A better picture comes from binning the sites
by minor allele frequency. Here is the concordance (missing counted as discordant) per bin.

In [ ]:
maf  <- pmin(true_freq, 1 - true_freq)
bins <- cut(maf, breaks=c(0, 0.01, 0.05, 0.1, 0.2, 0.5), include.lowest=TRUE)
print(table(bins))

# concordance per method (rows) per frequency bin (columns),
# counting a missing genotype as an error
conc_bins <- sapply(split(seq_along(maf), bins), function(idx)
    sapply(gt_list, function(x){
        x[is.na(x)] <- -1
        mean(res_truth_mat[idx, ] == x[idx, ], na.rm=TRUE)
    }))

# the same, but ignoring the missing genotypes
conc_bins_nonmis <- sapply(split(seq_along(maf), bins), function(idx)
    sapply(gt_list, function(x)
        mean(res_truth_mat[idx, ] == x[idx, ], na.rm=TRUE)))

cat("
missing counted as an error:
"); print(round(conc_bins, 3))
cat("
missing genotypes ignored:
");  print(round(conc_bins_nonmis, 3))

cols <- c("black", "darkorange", "steelblue", "forestgreen", "firebrick")
matplot(t(conc_bins), type="b", pch=16, lty=1, col=cols, lwd=2,
        xaxt="n", ylim=c(0, 1), ylab="concordance with truth",
        xlab="minor allele frequency bin",
        main="Genotype concordance across frequency bins",
        sub="solid + filled: missing counted as an error      dashed + open: missing genotypes ignored")
matlines(t(conc_bins_nonmis), type="b", pch=1, lty=2, col=cols, lwd=2)
axis(1, at=1:ncol(conc_bins), labels=colnames(conc_bins))
legend("bottomright", rownames(conc_bins), col=cols, lwd=2, pch=16, bty="n")


**Follow-up**

 - Solid lines count a missing genotype as an error, dashed lines ignore missing genotypes. For which
   methods do the two lines differ, and why?
 - Are there differences across the frequency bins? Which method degrades most for rare variants?
 - The rarest bin looks *better* than the common bins for the reference panel based methods. Is that
   because they are good, or because guessing "homozygous reference" is nearly always right there?
 - Do the non-imputation and the imputation methods have different strengths?
 - What do you think happens to these differences when the depth of coverage increases from 1x to 5x?
 - What happens when the reference panel is *not* closely related to the target samples (e.g. a
   European panel used to impute a Kenyan sample)?
 - **Extra:** redo the plot using dosage (DS) instead of the called genotype, or use the squared
   correlation between the truth and the dosage instead of concordance. Which measure do you find
   more informative for rare variants?

### Quiz

In [ ]:
from jupyterquiz import display_quiz
display_quiz('call_genotypes_quiz3.json')

## 6. Bonus: QUILT2 on NIPT data

In **non-invasive prenatal testing** (NIPT) one sequences cell-free DNA from the blood of a pregnant
woman. The sample is a mixture: most of the DNA comes from the mother, and a fraction *f* (the *fetal
fraction*, often 5-30%) comes from the fetus. So a "1x NIPT sample" is really a low depth mixture of
two related genomes.

If you treat it as one ordinary diploid sample you will get the mother roughly right and the fetus not
at all. QUILT2 has a dedicated `nipt` mode which models three haplotypes (the mother's two, plus the
paternally inherited fetal one) together with the fetal fraction.

The extra bam file we left out of the bamlist is a simulated NIPT sample at 1x with a fetal fraction
of 0.3. The mother is `NA12828`. We impute it in both modes and compare - reusing the prepared
reference from the QUILT2 run above, so this is fast.

In [ ]:
source ~/advBinfImputation/env.sh

ls bams/FAM80*.bam > CEU_nipt_bams.list
cat CEU_nipt_bams.list

In [ ]:
source ~/advBinfImputation/env.sh
# ~20 sec - we reuse the reference panel prepared in the QUILT2 run above

PREPARED=quilt2_1_ind/RData/QUILT_prepared_reference.chr20.2000001.5000000.RData
ls -l $PREPARED

rm -rf quilt2_fam_diploid
mkdir -p quilt2_fam_diploid

# ordinary diploid mode: pretend the mixture is a single individual
$QUILT2 \
    --method=diploid \
    --bamlist=CEU_nipt_bams.list \
    --prepared_reference_filename=$PREPARED \
    --output_filename=quilt2_fam_diploid/quilt.chr20.2000001.5000000.vcf.gz \
    --chr=chr20 \
    --regionStart=2000001 \
    --regionEnd=5000000 \
    --buffer=500000 \
    --nGen=100

In [ ]:
source ~/advBinfImputation/env.sh
# ~20 sec

PREPARED=quilt2_1_ind/RData/QUILT_prepared_reference.chr20.2000001.5000000.RData

rm -rf quilt2_fam_nipt
mkdir -p quilt2_fam_nipt

# nipt mode needs one extra input: the fetal fraction of each sample
echo 0.3 > quilt2_fam_nipt/ff.list   # we assume 30% fetal DNA

$QUILT2 \
    --method=nipt \
    --fflist=quilt2_fam_nipt/ff.list \
    --bamlist=CEU_nipt_bams.list \
    --prepared_reference_filename=$PREPARED \
    --output_filename=quilt2_fam_nipt/quilt.chr20.2000001.5000000.vcf.gz \
    --chr=chr20 \
    --regionStart=2000001 \
    --regionEnd=5000000 \
    --buffer=500000 \
    --nGen=100

Now compare how well the **mother's** genotypes were recovered in the two modes. In `nipt` mode the
maternal dosage is reported in the `MDS` field, which we round to a genotype.

In [ ]:
setwd(path.expand("~/advBinfImputation"))
region <- "chr20:2000001-5000000"

mother_id <- "NA12828"
fetus_id  <- "NA12815"   # not in the truth set - see the question below

res_truth_mother <- vcftable("vcfs/CEU_true_set.chr20.vcf.gz", region,
                             vartype="snps", samples=mother_id)

# nipt mode: maternal dosage (MDS) rounded to a genotype
res_nipt <- vcftable("quilt2_fam_nipt/quilt.chr20.2000001.5000000.vcf.gz", region,
                     vartype="snps", format="MDS")
res_nipt$samples <- mother_id
res_nipt$gt <- round(res_nipt$MDS, digits=0)
nipt_mother_mat <- find_match(res_nipt, res_truth_mother, name="nipt_mother")

# diploid mode: the ordinary genotype
res_diploid <- vcftable("quilt2_fam_diploid/quilt.chr20.2000001.5000000.vcf.gz", region,
                        vartype="snps")
res_diploid$samples <- mother_id
res_diploid$gt <- as.matrix(res_diploid$gt)
diploid_mother_mat <- find_match(res_diploid, res_truth_mother, name="diploid_mother")

estimate_gt_tools(list(
    diploid = as.matrix(diploid_mother_mat),
    nipt    = as.matrix(nipt_mother_mat)), as.matrix(res_truth_mother$gt))

**Follow-up**

 - Does modelling the mixture help for the mother? Would you expect a bigger or a smaller difference
   if the fetal fraction were 0.05 instead of 0.3?
 - What happens if you give QUILT2 a wrong fetal fraction? Try changing `0.3` in the cell above to
   `0.1` and rerun the two last cells.
 - **How would you impute the fetus?** The fetus inherits one haplotype from the mother and one from
   the father. Which of the two is harder to get right from this data, and why? (the fetal genotypes
   are in the `FDS`/fetal dosage fields of the nipt output)
 - The fetus `NA12815` is not in our truth set. What data would you need to properly evaluate the
   fetal imputation?